In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/tamirka/notebook8c09694358/__results__.html
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_classification.csv
/kaggle/input/notebooks/tamirka/notebook8c09694358/__notebook__.ipynb
/kaggle/input/notebooks/tamirka/notebook8c09694358/__output__.json
/kaggle/input/notebooks/tamirka/notebook8c09694358/custom.css
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/wallet_buy_sell_lifecycle.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/wallet_realized_pnl_trades.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/wallet_hold_stats.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/wallet_realized_pnl_stats.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/events_clean.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/price_events.pa

In [2]:
from pathlib import Path
import os

BASE = Path("/kaggle/input/notebooks/tamirka/notebook8c09694358")

print("BASE exists:", BASE.exists())

for root, dirs, files in os.walk(BASE):
    level = root.replace(str(BASE), "").count(os.sep)
    if level <= 2:
        print("\n", root)
        for d in dirs[:20]:
            print("  DIR:", d)
        for f in files[:20]:
            print("  FILE:", f)

BASE exists: True

 /kaggle/input/notebooks/tamirka/notebook8c09694358
  DIR: wallet_type_detector_cache_v2
  DIR: wallet_type_detector_exports_v2
  FILE: __results__.html
  FILE: wallet_type_classification.csv
  FILE: __notebook__.ipynb
  FILE: __output__.json
  FILE: custom.css

 /kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2
  DIR: lifecycle_checkpoints_fast
  FILE: wallet_buy_sell_lifecycle.parquet
  FILE: wallet_realized_pnl_trades.parquet
  FILE: wallet_hold_stats.parquet
  FILE: wallet_realized_pnl_stats.parquet
  FILE: events_clean.parquet
  FILE: price_events.parquet
  FILE: fb_first_buys.parquet

 /kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/lifecycle_checkpoints_fast
  FILE: lifecycle_part_00008.parquet
  FILE: lifecycle_part_00011.parquet
  FILE: lifecycle_part_00019.parquet
  FILE: lifecycle_part_00015.parquet
  FILE: lifecycle_part_00022.parquet
  FILE: lifecycle_part_00003.parquet
  FILE: lifecycle_par

In [3]:
from pathlib import Path
import shutil

BASE = Path("/kaggle/input/notebooks/tamirka/notebook8c09694358")

INPUT_CACHE = BASE / "wallet_type_detector_cache_v2"
WORK_CACHE = Path("/kaggle/working/wallet_type_detector_cache_v2")

INPUT_EXPORTS = BASE / "wallet_type_detector_exports_v2"
WORK_EXPORTS = Path("/kaggle/working/exports_token_events_copy")

if WORK_CACHE.exists():
    shutil.rmtree(WORK_CACHE)
shutil.copytree(INPUT_CACHE, WORK_CACHE)

if WORK_EXPORTS.exists():
    shutil.rmtree(WORK_EXPORTS)
shutil.copytree(INPUT_EXPORTS, WORK_EXPORTS)

print("cache restored:", WORK_CACHE)
print("exports restored:", WORK_EXPORTS)
print("cache files:", len(list(WORK_CACHE.rglob("*"))))
print("export files:", len(list(WORK_EXPORTS.rglob("*"))))

cache restored: /kaggle/working/wallet_type_detector_cache_v2
exports restored: /kaggle/working/exports_token_events_copy
cache files: 34
export files: 9


In [4]:
# ============================================================
# 02 — REBUILD CLEAN FIRST-SIGNAL DATASET
# Creates fresh_signals_clean.parquet
# One first BUY signal per token, age 5–120s
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

CACHE_DIR = Path("/kaggle/working/wallet_type_detector_cache_v2")
EXPORT_DIR = Path("/kaggle/working/wallet_type_detector_exports_v2")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Use already loaded events if available, otherwise load
try:
    events
except NameError:
    events = pd.read_parquet(CACHE_DIR / "events_clean.parquet")

events = events.copy()
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
events["launchTime"] = pd.to_datetime(events["launchTime"], errors="coerce")

SIGNAL_AGE_MIN = 5
SIGNAL_AGE_MAX = 120
FUTURE_WINDOW = 180
PRE_WINDOWS = [5, 10, 30, 60]

token_groups = {
    token: g.sort_values("timestamp").reset_index(drop=True)
    for token, g in events.groupby("tokenAddr", sort=False)
}

print("tokens:", len(token_groups))

def get_window(g, t0, sec_back):
    start = t0 - pd.Timedelta(seconds=sec_back)
    return g[(g["timestamp"] >= start) & (g["timestamp"] <= t0)]

def get_future(g, t0, sec_forward):
    end = t0 + pd.Timedelta(seconds=sec_forward)
    return g[(g["timestamp"] >= t0) & (g["timestamp"] <= end)]

def pre_features_clean(g, t0, entry_price):
    feats = {}

    for sec in PRE_WINDOWS:
        w = get_window(g, t0, sec)
        buys = w[w["eventType"] == "BUY"]
        sells = w[w["eventType"] == "SELL"]

        bvol = buys["amountSol"].sum()
        svol = sells["amountSol"].sum()

        feats[f"buyVol{sec}s"] = bvol
        feats[f"sellVol{sec}s"] = svol
        feats[f"ratio{sec}s"] = bvol / svol if svol > 0 else np.inf

        feats[f"buyCount{sec}s"] = len(buys)
        feats[f"sellCount{sec}s"] = len(sells)
        feats[f"uniqueBuyers{sec}s"] = buys["wallet"].nunique()
        feats[f"uniqueSellers{sec}s"] = sells["wallet"].nunique()

        if len(w) > 0 and entry_price > 0:
            first_price = w.iloc[0]["price"] if w.iloc[0]["price"] > 0 else entry_price
            max_price = w["price"].max()
            min_price = w["price"].min()

            feats[f"delta{sec}sPct"] = ((entry_price - first_price) / first_price) * 100 if first_price > 0 else 0
            feats[f"maxPrice{sec}s"] = max_price
            feats[f"minPrice{sec}s"] = min_price
            feats[f"drawdown{sec}sPct"] = ((entry_price - max_price) / max_price) * 100 if max_price > 0 else 0
            feats[f"recoveryFromMin{sec}sPct"] = ((entry_price - min_price) / min_price) * 100 if min_price > 0 else 0
        else:
            feats[f"delta{sec}sPct"] = 0
            feats[f"maxPrice{sec}s"] = entry_price
            feats[f"minPrice{sec}s"] = entry_price
            feats[f"drawdown{sec}sPct"] = 0
            feats[f"recoveryFromMin{sec}sPct"] = 0

        if len(buys) > 0 and bvol > 0:
            top = buys.groupby("wallet")["amountSol"].sum().sort_values(ascending=False)
            feats[f"top1BuyerShare{sec}s"] = top.iloc[0] / bvol * 100
            feats[f"top3BuyerShare{sec}s"] = top.head(3).sum() / bvol * 100
            feats[f"top5BuyerShare{sec}s"] = top.head(5).sum() / bvol * 100
            feats[f"largestBuyerShare{sec}s"] = top.iloc[0] / bvol * 100
        else:
            feats[f"top1BuyerShare{sec}s"] = 0
            feats[f"top3BuyerShare{sec}s"] = 0
            feats[f"top5BuyerShare{sec}s"] = 0
            feats[f"largestBuyerShare{sec}s"] = 0

    return feats

def future_outcome_clean(g, t0, entry_price):
    f = get_future(g, t0, FUTURE_WINDOW)
    out = {}

    if f.empty or entry_price <= 0:
        out["maxProfit180s"] = 0
        out["maxDrawdown180s"] = 0
        out["finalPct180s"] = 0
        for tp in [10, 20, 30, 50, 70, 100, 120, 150]:
            out[f"reach{tp}"] = False
        out["dead"] = True
        out["stopFirst20"] = False
        out["stopFirst30"] = False
        return out

    max_p = f["price"].max()
    min_p = f["price"].min()

    out["maxProfit180s"] = ((max_p - entry_price) / entry_price) * 100
    out["maxDrawdown180s"] = ((min_p - entry_price) / entry_price) * 100
    out["finalPct180s"] = ((f.iloc[-1]["price"] - entry_price) / entry_price) * 100

    for tp in [10, 20, 30, 50, 70, 100, 120, 150]:
        out[f"reach{tp}"] = out["maxProfit180s"] >= tp

    out["dead"] = out["maxProfit180s"] < 5

    stop20 = False
    stop30 = False

    for _, r in f.sort_values("timestamp").iterrows():
        p = ((r["price"] - entry_price) / entry_price) * 100

        if p <= -20:
            stop20 = True
        if p <= -30:
            stop30 = True

        if p >= 20:
            break

    out["stopFirst20"] = stop20
    out["stopFirst30"] = stop30

    return out

rows = []
tokens = list(token_groups.keys())

for idx, token in enumerate(tokens):
    if idx % 1000 == 0:
        print(f"processed {idx}/{len(tokens)}")

    g = token_groups[token]

    early = g[
        (g["ageSec"] >= SIGNAL_AGE_MIN) &
        (g["ageSec"] <= SIGNAL_AGE_MAX) &
        (g["eventType"] == "BUY")
    ]

    if early.empty:
        continue

    # first BUY signal only per token
    ev = early.iloc[0]
    t0 = ev["timestamp"]
    entry_price = ev["price"]

    if pd.isna(t0) or pd.isna(entry_price) or entry_price <= 0:
        continue

    feats = pre_features_clean(g, t0, entry_price)
    outcome = future_outcome_clean(g, t0, entry_price)

    row = {
        "tokenAddr": token,
        "entryTime": t0,
        "entryAgeSec": ev["ageSec"],
        "entryPrice": entry_price,
        "entryBuySol": ev["amountSol"],
        "entryWallet": ev["wallet"],
        "txSignature": ev["txSignature"],
        "slot": ev["slot"],
        "launchTime": ev["launchTime"],
    }

    row.update(feats)
    row.update(outcome)
    rows.append(row)

signals_clean = pd.DataFrame(rows)

print("signals_clean shape:", signals_clean.shape)
print("reach20:", signals_clean["reach20"].mean() * 100)
print("reach50:", signals_clean["reach50"].mean() * 100)
print("reach100:", signals_clean["reach100"].mean() * 100)
print("reach120:", signals_clean["reach120"].mean() * 100)
print("dead:", signals_clean["dead"].mean() * 100)
print("stopFirst20:", signals_clean["stopFirst20"].mean() * 100)
print("stopFirst30:", signals_clean["stopFirst30"].mean() * 100)

signals_clean.to_parquet(CACHE_DIR / "fresh_signals_clean.parquet", index=False)
print("saved:", CACHE_DIR / "fresh_signals_clean.parquet")

tokens: 157651
processed 0/157651
processed 1000/157651
processed 2000/157651
processed 3000/157651
processed 4000/157651
processed 5000/157651
processed 6000/157651
processed 7000/157651
processed 8000/157651
processed 9000/157651
processed 10000/157651
processed 11000/157651
processed 12000/157651
processed 13000/157651
processed 14000/157651
processed 15000/157651
processed 16000/157651
processed 17000/157651
processed 18000/157651
processed 19000/157651
processed 20000/157651
processed 21000/157651
processed 22000/157651
processed 23000/157651
processed 24000/157651
processed 25000/157651
processed 26000/157651
processed 27000/157651
processed 28000/157651
processed 29000/157651
processed 30000/157651
processed 31000/157651
processed 32000/157651
processed 33000/157651
processed 34000/157651
processed 35000/157651
processed 36000/157651
processed 37000/157651
processed 38000/157651
processed 39000/157651
processed 40000/157651
processed 41000/157651
processed 42000/157651
processed